In [1]:
import os
import random
import sys

import numpy as np
import torch
from torch import nn

sys.path.append(os.path.abspath("./"))

SEED = 12
random.seed(SEED)
np.random.seed(SEED)

from codebooks_defs_generator import generate_cb_definitions
from input_image_gen import generate_gemm_input_file
from gemm_layer_generator import generate_template_gemm, generate_gemm_data_file


This notebook mirrors the LeNet generator flow for Transformer GEMM layers:
1. Generate `codebooks_def.h`
2. Generate `input_matrix.h`
3. Generate one `gemm_header_<id>.h` per GEMM layer
4. Generate `gemm_data.h` to include all generated headers

In [2]:
N_LEARNERS = 4
CODEBOOK_SIZE = 8
SVE_LANES = 1
SAME_SEQ = True

USE_BIAS = True
USE_F16 = False
USE_CODEBOOKS = True

SEQ_LEN = 4
INPUT_SIZE = 16
TILE_SIZE = 5

gemm_0 = {"type": "gemm", "out_size": 4}
# gemm_1 = {"type": "gemm", "out_size": 16}

GEMM_STRUCTURE = [gemm_0]

OUT_FOLDER = "./../gemm_definitions/"
os.makedirs(OUT_FOLDER, exist_ok=True)


In [3]:
generate_cb_definitions(
    OUT_FOLDER + "codebooks_def.h",
    N_LEARNERS,
    CODEBOOK_SIZE,
    SVE_LANES,
    USE_BIAS,
    USE_F16,
    SAME_SEQ,
    USE_CODEBOOKS,
)

input_matrix = generate_gemm_input_file(
    OUT_FOLDER + "input_matrix.h",
    SEQ_LEN,
    INPUT_SIZE,
    USE_F16,
)

network = [nn.ModuleDict({}) for _ in range(N_LEARNERS)]

in_size = INPUT_SIZE
gemm_values = []
gemm_biases = []

for lay_cnt, layer in enumerate(GEMM_STRUCTURE):
    print("[{}] {}".format(lay_cnt, layer["type"]))
    print("\t", layer)

    if layer["type"] != "gemm":
        print("ERROR!")
        raise SystemExit(1)

    out_size = layer["out_size"]
    in_shape = (SEQ_LEN, in_size)
    out_shape = (SEQ_LEN, out_size)

    layer_values, layer_biases = generate_template_gemm(
        SAME_SEQ,
        OUT_FOLDER + "gemm_header_{}.h".format(lay_cnt),
        lay_cnt,
        N_LEARNERS,
        CODEBOOK_SIZE,
        TILE_SIZE,
        in_size,
        out_size,
        USE_F16,
        USE_CODEBOOKS,
    )

    gemm_values.append(layer_values)
    gemm_biases.append(layer_biases)

    for learner in range(N_LEARNERS):
        linear = nn.Linear(in_size, out_size, bias=USE_BIAS)
        weight = torch.tensor(np.asarray(layer_values[learner]).reshape(out_size, in_size), dtype=torch.float32)
        with torch.no_grad():
            linear.weight.copy_(weight)
            if USE_BIAS:
                linear.bias.copy_(torch.tensor(layer_biases[learner], dtype=torch.float32))
        network[learner][f"gemm_{lay_cnt}"] = linear

    print("In shape:", in_shape)
    print("Out shape:", out_shape)
    print()

    in_size = out_size

generate_gemm_data_file(OUT_FOLDER + "gemm_data.h", len(GEMM_STRUCTURE))

print("Generated files in", OUT_FOLDER)
print("Input matrix:")
print(input_matrix)


[0] gemm
	 {'type': 'gemm', 'out_size': 4}


TypeError: generate_template_gemm() missing 2 required positional arguments: 'use_codebooks' and 'use_bias'

In [ ]:
input_tensor = torch.tensor(input_matrix, dtype=torch.float32)
print("input shape:", input_tensor.shape)

for ens in range(N_LEARNERS):
    print(f"\n=============== LEARNER {ens} ===============\n")

    y = input_tensor
    for layer_name, layer_module in network[ens].items():
        y = layer_module(y)
        print(f"{layer_name} output shape:", y.shape)
        print(y)

    print("Final output:")
    print(y)


input shape: torch.Size([4, 16])

=============== LEARNER 0 ===============

gemm_0 output shape: torch.Size([4, 4])
tensor([[0.2088, 0.3171, 0.0907, 0.9097],
        [0.1712, 0.4959, 0.2312, 0.7819],
        [0.1147, 0.3259, 0.2613, 0.7291],
        [0.2188, 0.3741, 0.0685, 0.7612]], grad_fn=<AddmmBackward0>)
Final output:
tensor([[0.2088, 0.3171, 0.0907, 0.9097],
        [0.1712, 0.4959, 0.2312, 0.7819],
        [0.1147, 0.3259, 0.2613, 0.7291],
        [0.2188, 0.3741, 0.0685, 0.7612]], grad_fn=<AddmmBackward0>)
